<a href="https://colab.research.google.com/github/erickvaldezsallagos/ingenieria/blob/main/limpia_de_bases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import re

# ==========================================
# 1. LIMPIEZA DE LA BASE DE DATOS DE APPLE
# ==========================================
print("Procesando base de datos de Apple...")
df_apple = pd.read_csv('apple_mx_productos.csv')

# Eliminar columnas 100% vacías que no aportan valor
empty_cols_apple = ['sku', 'gtin', 'calificacion', 'resenas', 'condicion']
df_apple = df_apple.drop(columns=[c for c in empty_cols_apple if c in df_apple.columns])

# Limpieza de espacios en textos
df_apple['nombre'] = df_apple['nombre'].astype(str).str.strip()
df_apple['descripcion'] = df_apple['descripcion'].astype(str).str.strip()

# Filtrar registros sin nombre o URL válida
df_apple = df_apple.dropna(subset=['nombre', 'url'])
df_apple = df_apple[(df_apple['nombre'].str.lower() != 'nan') & (df_apple['nombre'] != '')]

# DEDUPLICACIÓN INTELIGENTE (Protegiendo variantes de color, precio y versión mediante URL única)
df_apple_clean = df_apple.drop_duplicates(subset=['url', 'color', 'precio'], keep='first').copy()
df_apple_clean['marca'] = 'Apple'

print(f"-> Apple limpio: {len(df_apple_clean)} registros (variantes preservadas).")


# ==========================================
# 2. LIMPIEZA DE LA BASE DE DATOS DE SAMSUNG
# ==========================================
print("\nProcesando base de datos de Samsung...")
xls_samsung = pd.ExcelFile('samsung_mx_productos.xlsx')
df_samsung = pd.read_excel(xls_samsung, sheet_name=xls_samsung.sheet_names[0])

# Eliminar columnas 100% vacías
empty_cols_samsung = ['gtin', 'calificacion', 'resenas', 'color', 'condicion']
df_samsung = df_samsung.drop(columns=[c for c in empty_cols_samsung if c in df_samsung.columns])

# Limpiar etiquetas HTML residuales en los textos
df_samsung['nombre'] = df_samsung['nombre'].astype(str).str.replace(r'<[^>]*>', '', regex=True).str.strip()
df_samsung['descripcion'] = df_samsung['descripcion'].astype(str).str.replace(r'<[^>]*>', '', regex=True).str.strip()

# Normalizar precios (descartando texto basura tipo "Quantity: 3 - £120.00")
def parse_price(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if 'Quantity' in val_str or '£' in val_str:
        return None
    try:
        return float(val_str)
    except ValueError:
        return None

df_samsung['precio_numerico'] = df_samsung['precio'].apply(parse_price)

# Filtrar registros sin nombre o URL válida
df_samsung = df_samsung.dropna(subset=['nombre', 'url'])
df_samsung = df_samsung[(df_samsung['nombre'].str.lower() != 'nan') & (df_samsung['nombre'] != '')]

# DEDUPLICACIÓN INTELIGENTE (Protegiendo variantes y modelos distintos)
df_samsung_clean = df_samsung.drop_duplicates(subset=['url', 'precio_numerico', 'modelo'], keep='first').copy()
df_samsung_clean['marca'] = 'Samsung'

print(f"-> Samsung limpio: {len(df_samsung_clean)} registros (variantes preservadas).")


# ==========================================
# 3. UNIÓN CONJUNTA (CREACIÓN DEL ARCHIVO MAESTRO)
# ==========================================
common_cols = ['marca', 'nombre', 'modelo', 'precio', 'precio_numerico', 'moneda', 'disponibilidad', 'categoria', 'url', 'descripcion', 'color', 'imagen']

df_a_sub = df_apple_clean[[c for c in common_cols if c in df_apple_clean.columns]].copy()
df_s_sub = df_samsung_clean[[c for c in common_cols if c in df_samsung_clean.columns]].copy()

# Concatenar ambos catálogos limpios
df_maestro = pd.concat([df_a_sub, df_s_sub], ignore_index=True)

print("\n==========================================")
print(f"¡PROCESO FINALIZADO CON ÉXITO!")
print(f"Total de registros maestros consolidados: {len(df_maestro)}")
print(df_maestro['marca'].value_counts())
print("==========================================")

# Exportar archivo final listo para el proyecto
output_file = 'catalogo_tecnologia_maestro_limpio.csv'
df_maestro.to_csv(output_file, index=False)
print(f"\nArchivo guardado exitosamente como: {output_file}")

Procesando base de datos de Apple...
-> Apple limpio: 3863 registros (variantes preservadas).

Procesando base de datos de Samsung...
-> Samsung limpio: 3432 registros (variantes preservadas).

¡PROCESO FINALIZADO CON ÉXITO!
Total de registros maestros consolidados: 7295
marca
Apple      3863
Samsung    3432
Name: count, dtype: int64

Archivo guardado exitosamente como: catalogo_tecnologia_maestro_limpio.csv


In [7]:
import pandas as pd

# 1. Cargar el archivo maestro actual
df = pd.read_csv('catalogo_tecnologia_maestro_limpio.csv')

print(f"Registros antes de afinar: {len(df)}")

# 2. Filtrar páginas de categorías genéricas, banners o índices de navegación del sitio web
# (Estas filas suelen llamarse "Accesorios para...", "Watch Bands Main...", etc.)
df_refined = df[~df['nombre'].str.startswith('Accesorios para', na=False)].copy()
df_refined = df_refined[~df_refined['nombre'].str.contains('Watch Bands Main', na=False)].copy()

print(f"¡Proceso completado!")
print(f"Registros finales refinados: {len(df_refined)}")
print("\nDesglose por marca:")
print(df_refined['marca'].value_counts())

# 3. Guardar el archivo maestro optimizado definitivo
output_file = 'catalogo_tecnologia_maestro_5500_refinado.csv'
df_refined.to_csv(output_file, index=False)
print(f"\nArchivo guardado exitosamente como: {output_file}")


Registros antes de afinar: 7295
¡Proceso completado!
Registros finales refinados: 6035

Desglose por marca:
marca
Samsung    3432
Apple      2603
Name: count, dtype: int64

Archivo guardado exitosamente como: catalogo_tecnologia_maestro_5500_refinado.csv
